# Fairness OT — Visualization Notebook

Edit this notebook to iterate on plots. Each section loads data and produces figures.
Results CSVs are in the `results/` directory.

F1: individual fairness. F2: group fairness. F3: combined group + individual (enhanced stable solver).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import ast

RESULTS_DIR = Path.cwd() / 'results'

# ── Tweaks you can edit ──────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize': (7, 4.5),
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
})

DATASET_COLORS = {
    'adult': '#1f77b4',
    'german': '#ff7f0e',
    'compas': '#2ca02c',
    'lsac': '#d62728',
    'credit_default': '#9467bd',
    'heart_disease': '#8c564b',
    'saheart': '#e377c2',
    'student': '#7f7f7f',
    'acsincome': '#bcbd22',
    'communities': '#17becf',
    'hmda': '#aec7e8',
}

DATASET_LABELS = {
    'adult': 'Adult',
    'german': 'German Credit',
    'compas': 'COMPAS',
    'lsac': 'LSAC',
    'credit_default': 'Credit Default',
    'heart_disease': 'Heart Disease',
    'saheart': 'SA Heart',
    'student': 'Student',
    'acsincome': 'ACSIncome',
    'communities': 'Communities',
    'hmda': 'HMDA',
}

# ── Demographic group labels for theta in F2 ─────────────────
# Indices correspond to theta_0, theta_1, ... from the solver output.
# For multi-class attributes, leave as None (falls back to 'Group i').
GROUP_LABELS = {
    'german': ['Female', 'Male'],
    'adult': ['Female', 'Male'],
    'compas': ['African-American', 'Asian', 'Caucasian', 'Hispanic', 'Native American', 'Other'],
    'lsac': ['Non-white', 'White'],
    'credit_default': ['Female', 'Male'],
    'saheart': ['Absent fam.', 'Present fam.'],
    'student': ['Female', 'Male'],
    'communities': ['Non-white maj.', 'White maj.'],
    'heart_disease': ['Female', 'Male'],
    'acsincome': ['Female', 'Male'],
}


def load_csv(name):
    path = RESULTS_DIR / f'{name}.csv'
    if not path.exists():
        print(f'  WARNING: {path} not found')
        return pd.DataFrame()
    return pd.read_csv(path)


def savefig(fig, name):
    path = RESULTS_DIR / f'{name}.pdf'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'  Saved {path}')
    plt.show()


def parse_m_z(mz_str):
    try:
        return ast.literal_eval(mz_str) if isinstance(mz_str, str) else {}
    except:
        return {}

---
## F1 — Individual Fairness

In [ ]:
df = load_csv('f1_individual_fairness')
df['DATASET'] = df['dataset'].replace(DATASET_LABELS)

In [ ]:
# F1 — Runtime heatmap (actual values with std, colored by row-relative)
df = df[df['DATASET'] != 'Communities']
if not df.empty:
    grp = df.groupby(['DATASET', 'lambda_ind'])['execution_time'].agg(['mean', 'std']).reset_index()
    means = grp.pivot_table(index='DATASET', columns='lambda_ind', values='mean', aggfunc='first')
    stds = grp.pivot_table(index='DATASET', columns='lambda_ind', values='std', aggfunc='first')
    colors = means.div(means.max(axis=1), axis=0)
    annot = means.round(2).astype(str) + '\n±\n' + stds.round(2).astype(str)
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(colors, annot=annot, fmt='', cmap='YlOrRd', ax=ax,
                cbar_kws={'label': 'Relative runtime'}, annot_kws={'size': 8})
    ax.set_xlabel(r'$\lambda_{I}$', size=20)
    ax.set_ylabel('Dataset', size=20)
    ax.set_title('Individual Fairness formulation runtime (s)', size=20)
    savefig(fig, 'individual_runtime')
    plt.close(fig)

In [ ]:
# F1 — Dual y-axis: cost (left) + per-individual variance (right), both vs lambda
if not df.empty:
    for ds in df['dataset'].unique():
        fig, ax1 = plt.subplots()
        ax2 = ax1.twinx()
        sub = df[df['dataset'] == ds].groupby('lambda_ind')[['avg_cost', 'per_individual_var']].agg(['mean', 'std']).reset_index()
        sub.columns = ['lambda_ind', 'cost_mean', 'cost_std', 'var_mean', 'var_std']
        if 'credit' in ds:
            sub = sub[sub['lambda_ind'] > 0.02]
        ax1.errorbar(sub['lambda_ind'], sub['cost_mean'], yerr=sub['cost_std'],
                      marker='o', color='blue', label='Cost', capsize=3, alpha=0.8)
        ax2.errorbar(sub['lambda_ind'], sub['var_mean'], yerr=sub['var_std'],
                      marker='o', color='red', label='Var.', capsize=3, alpha=0.8)
        ax1.set_xlabel(r'$\lambda_{I}$', size=18)
        ax1.set_ylabel('Average Transport Cost', size=18)
        ax2.set_ylabel('Variance', size=18)
        ax1.set_xscale('symlog')
        ax1.set_title(f'{DATASET_LABELS.get(ds, ds)} - Cost and Individual Variance', size=20)
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax2.legend(lines1 + lines2, labels1 + labels2, loc='best', fontsize=14)
        fig.tight_layout()
        savefig(fig, f'individual_fairness_dual_{ds}')
        plt.close(fig)

---
## F2 — Group Fairness

In [ ]:
df3 = load_csv('f2_group_fairness')
df3['DATASET'] = df3['dataset'].replace(DATASET_LABELS)
if not df3.empty:
    for ds in df3['dataset'].unique():
        fig, ax1 = plt.subplots()
        ax2 = ax1.twinx()
        sub = df3[df3['dataset'] == ds].groupby('lambda_g')[['avg_cost', 'barycentric_w2_sum', 'group_var']].agg(['mean', 'std']).reset_index()
        sub = sub[sub['lambda_g'] > 0.1]
        sub.columns = ['lambda_g', 'cost_mean', 'cost_std', 'barycentric_w2_mean', 'barycentric_w2_std', 'var_mean', 'var_std']
        color = DATASET_COLORS.get(ds, '#333')
        ax1.errorbar(sub['lambda_g'], sub['cost_mean'], yerr=sub['cost_std'],
                      marker='o', color='blue', label='Cost', capsize=3, alpha=0.8)
#        ax1.errorbar(1/sub['lambda_g'], sub['barycentric_w2_mean'], yerr=sub['barycentric_w2_std'],
#                      marker='o', color='green', label='Wass. cost')
        ax2.errorbar(sub['lambda_g'], sub['var_mean'], yerr=sub['var_std'],
                      marker='o', color='red', label='Var.', capsize=3, alpha=0.8)
        ax1.set_xlabel(r'$\lambda_G$', size=18)
        ax1.set_ylabel('Average Transport Cost', size=18)
        ax2.set_ylabel('Group Variance', size=18)
        ax1.set_xscale('symlog')
        ax1.set_title(f'{DATASET_LABELS.get(ds, ds)} - Cost and Group Variance', size=20)
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax2.legend(lines1 + lines2, labels1 + labels2, loc='best', fontsize=14)
        fig.tight_layout()
#        savefig(fig, f'e3_group_fairness_dual_inverse_{ds}')
        savefig(fig, f'group_fairness_dual_{ds}')
        plt.close(fig)

In [ ]:
# F2 — Runtime heatmap (actual values with std, colored by row-relative)
df3['DATASET'] = df3['dataset'].replace(DATASET_LABELS)
df3 = df3[df3['DATASET'] != 'Communities']
df3 = df3[df3['lambda_g'] >= 0.1]
if not df3.empty:
    grp = df3.groupby(['DATASET', 'lambda_g'])['execution_time'].agg(['mean', 'std']).reset_index()
    means = grp.pivot_table(index='DATASET', columns='lambda_g', values='mean', aggfunc='first')
    stds = grp.pivot_table(index='DATASET', columns='lambda_g', values='std', aggfunc='first')
    colors = means.div(means.max(axis=1), axis=0)
    annot = means.round(2).astype(str) + '\n±' + stds.round(2).astype(str)
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(colors, annot=annot, fmt='', cmap='YlOrRd', ax=ax,
                cbar_kws={'label': 'Relative runtime'}, annot_kws={'size': 7})
    ax.set_xlabel(r'$\lambda_G$', size=20)
    ax.set_ylabel('Dataset', size=20)
    ax.set_title('Group fairness formulation runtime (s)', size=20)
    savefig(fig, 'group_fairness_runtime_heatmap')
    plt.close(fig)

In [ ]:
# F2 — Theta vs lambda_g for all demographic groups
df3 = df3[df3['lambda_g'] >= 0.1]
if not df3.empty:
    theta_cols = [c for c in df3.columns if c.startswith('theta_')]
    if theta_cols:
        n_groups = len(theta_cols)
        palette = sns.color_palette('Set1', n_colors=n_groups)
        

        for ds in df3['dataset'].unique():
            fig, ax = plt.subplots()  
            sub = df3[df3['dataset'] == ds].groupby('lambda_g')[theta_cols].agg(['mean', 'std'])
            if 'compas' in ds.lower():
                print(sub)
                sub = sub[sub.index >= 0.2]
            group_labels = GROUP_LABELS.get(ds, [f'Group {i}' for i in range(n_groups)])

            if ds.lower() != 'compas':
                theta_cols_data = theta_cols[:2]
            else:
                theta_cols_data = theta_cols

            for i, col in enumerate(theta_cols_data):
                means = sub[(col, 'mean')]
                stds = sub[(col, 'std')]
                label = group_labels[i] if i < len(group_labels) else f'Group {i}'
                if i != 5:
                    color = palette[i]
                else:
                    color = 'black'
                ax.errorbar(means.index, means, yerr=stds,
                            marker='o', capsize=3, color=color,
                            label=label)
            ax.set_xlabel(r'$\lambda_G$', size=20)
            ax.set_ylabel(r'$\theta$', size=20)
            ax.set_xscale('symlog')
            ax.set_title(rf'{DATASET_LABELS.get(ds, ds)} - Penalization factor ($\theta$)', size=20)
            ax.legend(fontsize=13)
            savefig(fig, f'group_fairness_theta_{ds}')
            plt.close(fig)
    else:
        print('  No theta columns found — re-run F2 with updated runner.py')

---
## F3 — Combined Group + Individual

In [ ]:
# F3 (enhanced mixed solver) — heatmaps from the new stable-formulation results.
# Reads results/f3_mixed_fairness.csv (written by f3_mixed_fairness.py).
df_f3 = load_csv('f3_mixed_fairness')

if not df_f3.empty:
    for col, label, cmap, fmt in [
        ('avg_cost', 'Avg. Cost', 'YlOrRd', '.2f'),
        ('per_individual_var', 'Individual Variance', 'YlOrRd', '.2f'),
        ('group_var', 'Group Variance', 'YlOrRd', '.2f'),
    ]:
        for ds in df_f3['dataset'].unique():
            sub_mean = df_f3[df_f3['dataset'] == ds].groupby(['lambda_ind', 'lambda_g'])[col].mean().reset_index()
            sub_std = df_f3[df_f3['dataset'] == ds].groupby(['lambda_ind', 'lambda_g'])[col].std().reset_index()

            sub_mean = sub_mean[(sub_mean['lambda_g'] == 0) | (sub_mean['lambda_g'] >= 0.1)]
            sub_std = sub_std[(sub_std['lambda_g'] == 0) | (sub_std['lambda_g'] >= 0.1)]

            pivot_mean = sub_mean.pivot(index='lambda_g', columns='lambda_ind', values=col)
            pivot_std = sub_std.pivot(index='lambda_g', columns='lambda_ind', values=col)

            annot_matrix = np.empty_like(pivot_mean.values, dtype=object)
            for i in range(pivot_mean.shape[0]):
                for j in range(pivot_mean.shape[1]):
                    m = pivot_mean.values[i, j]
                    s = pivot_std.values[i, j]
                    annot_matrix[i, j] = f"{m:{fmt}}\n±\n{s:{fmt}}"

            fig, ax = plt.subplots(figsize=(7, 6))
            sns.heatmap(pivot_mean, annot=annot_matrix, fmt='', cmap=cmap, annot_kws={'size': 7}, ax=ax)
            ax.set_xlabel(r'$\lambda_{I}$', size=20)
            ax.set_ylabel(r'$\lambda_G$', size=20)
            ax.set_title(f'{DATASET_LABELS.get(ds, ds)}: {label}', size=20)

            savefig(fig, f'f3_enhanced_{ds}_{col}')
            plt.close(fig)

    # Runtime heatmap (actual values with std, colored by row-relative)
    df_f3_rt = df_f3[(df_f3['lambda_g'] > 0.09) | (df_f3['lambda_g'] == 0)]
    for ds in df_f3_rt['dataset'].unique():
        grp = df_f3_rt[df_f3_rt['dataset'] == ds].groupby(['lambda_g', 'lambda_ind'])['execution_time'].agg(['mean', 'std']).reset_index()
        means = grp.pivot_table(index='lambda_g', columns='lambda_ind', values='mean', aggfunc='first')
        stds = grp.pivot_table(index='lambda_g', columns='lambda_ind', values='std', aggfunc='first')
        colors = means.div(means.max(axis=1), axis=0)
        annot = means.round(2).astype(str) + '\n±\n' + stds.round(2).astype(str)
        fig, ax = plt.subplots(figsize=(10, 6))
        sns.heatmap(colors, annot=annot, fmt='', cmap='YlOrRd', ax=ax, annot_kws={'size': 7},
                    cbar_kws={'label': 'Relative Runtime (row-normalized)'})
        ax.set_xlabel(r'$\lambda_{I}$', size=20)
        ax.set_ylabel(r'$\lambda_{G}$', size=20)
        ax.set_title(f'{DATASET_LABELS.get(ds, ds)} — Runtime (s) Heatmap', size=20)
        savefig(fig, f'f3_enhanced_{ds}_combined_runtime_heatmap')
        plt.close(fig)
